In [45]:
from datasets import load_from_disk
import numpy as np
from tqdm import tqdm
import json
from ml4setk.Parsing.Code.TreeSitterQuery import TreeSitterQuery
from transformers import AutoTokenizer
import os
import time
import random


In [47]:
# Load local dataset
ds = load_from_disk("../data/Java/satd-100k-counts")
print(len(ds))
ds

# Select 10,000 random indices
random_indices = random.sample(range(len(ds)), 10_000)

# Select the corresponding rows from the dataset
random_ds = ds.select(random_indices)
print(len(random_ds))

100000
10000


In [48]:
output_dir = "../data/thresholds"
os.makedirs(output_dir, exist_ok=True)


def save_threshold(stats, filepath):
    with open(os.path.join(output_dir, filepath), "w") as f:
        json.dump(stats, f, indent=4)

def load_threshold(filepath):
    with open(os.path.join(output_dir, filepath), "r") as f:
        data = json.load(f)
    return data

In [49]:
# Tokenizer and model setup
checkpoint_smol = "HuggingFaceTB/SmolLM2-135M"
tokenizer_smol = AutoTokenizer.from_pretrained(checkpoint_smol)

checkpoint_star = "bigcode/starcoder2-3b"
tokenizer_star = AutoTokenizer.from_pretrained(checkpoint_star)

language = "java"
tree_query = TreeSitterQuery(language=language)

# Define models and their tokenizers
models = {
    "SmolLM135M": tokenizer_smol,
    "StarCoder2_3B": tokenizer_star,
}

c:\Users\lucaw\TUProjects\RP\ML4SE-toolkit\.venv\Lib\site-packages\tree_sitter\__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [50]:
def calculate_stats(lengths):
    """
    Calculate mean, standard deviation, and threshold for a list of lengths.
    """
    mean_len = np.mean(lengths)
    std_len = np.std(lengths)
    threshold = mean_len + 2 * std_len
    return mean_len, std_len, threshold

In [56]:
def get_comment_stats(dataset):
    lengths = {}
    for model_name, _ in models.items():
        lengths[model_name] = {
            "line": [],
            "block": [],
            "all": [],
        }

    for file in tqdm(dataset, miniters=10000, desc="Processing comments"):
        content = file["content"]
        
        # Parse line comments
        line_comments = tree_query.parse(content, '(line_comment) @comment')
        for (_, _, comment) in line_comments:
            for model_name, tokenizer in models.items():
                len_tokens = len(tokenizer(comment, return_tensors="pt")["input_ids"][0])
                lengths[model_name]["line"].append(len_tokens)

        # Parse block comments
        block_comments = tree_query.parse(content, '(block_comment) @comment')
        for (_, _, comment) in block_comments:
            for model_name, tokenizer in models.items():
                len_tokens = len(tokenizer(comment, return_tensors="pt")["input_ids"][0])
                lengths[model_name]["block"].append(len_tokens)

    results = {}
    for model_name, _ in models.items():
        line_mean, line_std, line_threshold = calculate_stats(lengths[model_name]["line"])
        block_mean, block_std, block_threshold = calculate_stats(lengths[model_name]["block"])
        all_mean, all_std, all_threshold = calculate_stats(lengths[model_name]["line"] + lengths[model_name]["block"])

        results[model_name] = {
            "line": {
                "mean": line_mean,
                "std": line_std,
                "threshold": line_threshold
            },
            "block": {
                "mean": block_mean,
                "std": block_std,
                "threshold": block_threshold
            },
            "all": {
                "mean": all_mean,
                "std": all_std,
                "threshold": all_threshold
            },
        }
    return results

In [57]:
# Get stats for line, block, and all comments combined
comment_stats = get_comment_stats(ds)

# Save the stats to separate files
for model_name, stats in comment_stats.items():
    save_threshold(stats, f"comment_stats_100k_{model_name}.json")

Processing comments: 100%|██████████| 100000/100000 [11:40<00:00, 142.67it/s] 


In [53]:
data = load_threshold("comment_method_stats_10k_SmolLM135M.json")
print(data)
print("Line comment threshold:", data["line"]["threshold"])

{'line': {'mean': 17.66737014092178, 'std': 9.910597605155614, 'threshold': 37.48856535123301}, 'block': {'mean': 38.4312858284995, 'std': 28.828836453620017, 'threshold': 96.08895873573954}, 'all': {'mean': 27.28693359042704, 'std': 23.897210076401674, 'threshold': 75.08135374323038}, 'method': {'mean': 68.83492549402642, 'std': 235.50073071974714, 'threshold': 539.8363869335207}}
Line comment threshold: 37.48856535123301


In [61]:
def get_method_stats(dataset):
    lengths = {}
    for model_name, _ in models.items():
        lengths[model_name] = {
            "method": []
        }

    for file in tqdm(dataset, miniters=1000, maxinterval=float('inf'), desc="Processing comments"):
        content = file["content"]

        # Parse methods
        methods = tree_query.parse(content, '(method_declaration) @decl')
        for (_, _, method) in methods:
            for model_name, tokenizer in models.items():
                len_tokens = len(tokenizer(method, return_tensors="pt")["input_ids"][0])
                lengths[model_name]["method"].append(len_tokens)

    results = {}
    for model_name, _ in models.items():
        method_mean, method_std, method_threshold = calculate_stats(lengths[model_name]["method"])

        results[model_name] = {
            "method": {
                "mean": method_mean,
                "std": method_std,
                "threshold": method_threshold
            }
        }
    return results

In [ ]:
# get stats for methods
method_stats = get_method_stats(ds)

# Save the stats to separate files
for model_name, stats in method_stats.items():
    save_threshold(stats, f"method_stats_100k_{model_name}.json")

Processing comments:   3%|▎         | 2968/100000 [00:43<23:27, 68.95it/s] 


KeyboardInterrupt: 

In [11]:
# get 10 random files and print their content
def print_random_files(dataset, n):
    for i in range(n):
        # Randomly select an index
        i = np.random.randint(0, len(dataset))
        file = dataset[i]
        print(f"File {i+1}: {file['file_name']}")
        print(file["content"])
        methods = tree_query.parse(file["content"], '(method_declaration) @decl')
        print(f"Number of methods: {len(methods)}")
        for _, _, method in methods:
            print(method)

In [15]:
print_random_files(ds, 1)

File 33481: DefaultAgentConnection.java
/*
 * Copyright (C) 2019 Qunar, Inc.
 *
 * This program is free software: you can redistribute it and/or modify
 * it under the terms of the GNU General Public License as published by
 * the Free Software Foundation, either version 3 of the License, or
 * (at your option) any later version.
 *
 * This program is distributed in the hope that it will be useful,
 * but WITHOUT ANY WARRANTY; without even the implied warranty of
 * MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
 * GNU General Public License for more details.
 *
 * You should have received a copy of the GNU General Public License
 * along with this program.  If not, see <https://www.gnu.org/licenses/>.
 */

package qunar.tc.bistoury.proxy.communicate.agent;

import io.netty.channel.Channel;
import qunar.tc.bistoury.proxy.communicate.AbstractConnection;

import java.util.Objects;

/**
 * @author zhenyu.nie created on 2019 2019/5/13 19:45
 */
public class DefaultAgentConne